# 5. Model Deployment Simulation - Live Production Forward Test
**Period: 2026-05-26 to 2026-05-30**

This notebook functions as the final staging and deployment simulation framework for the predictive pricing engine. After establishing our locked training parameters and cross-validation performance thresholds, we transition the Random Forest Regressor into a blind production loop. By evaluating the model on fresh, out-of-sample operational observations, this step mimics a live operational state to observe how cleanly our pricing trajectories generalise when confronted with real-world dynamic scaling and temporal variations.

## Objectives:
- Simulate Live Deployment: Test the trained model against fresh, out-of-sample data to measure its true predictive performance in a real-world scenario.
- Evaluate Operational Accuracy: Track baseline performance and error metrics (MAE, RMSE, $R^2$, and MAPE) on new data to monitor how well the model generalises over time.
- Retraining Cadence: Analyze potential changes in performance on live data to mathematically support future roadmap strategies, such as a monthly model re-training loop.

### Environment Setup

In [11]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error

In [2]:
# Load base model pkl files
model = joblib.load('flight_predictor_rf.joblib')
model_schema = joblib.load('model_feature_schema.joblib')

In [3]:
# Load dataset
df = pd.read_csv('./dataset.csv')

### Data Processing

In [4]:
# Handle date formatting and sorting them chronologically
df['date'] = pd.to_datetime(df['date'],format='mixed',dayfirst=True, errors='coerce')
df = df.sort_values('date').reset_index(drop = True)

In [5]:
# Isolate entries that were run on base model
cutoff_date = pd.to_datetime('2026-05-26')
df_new = df[df['date'] > cutoff_date].copy()

In [6]:
# Check if there are new entries after cutoff date
if len(df_new) == 0:
    print(f"No new data collected after {cutoff_date.strftime('%Y-%m-%d')}")
else: 
    print(f"Successfully isolated {len(df_new)} new flight records for inference calculation.")

Successfully isolated 480 new flight records for inference calculation.


In [7]:
# Apply same engineering steps as the model

# Boolean sanitisation - ensure that all boolean fields are shown as 0 or 1. 
for col in ['is_weekend', 'is_lcc', 'is_holiday_sin', 'is_holiday_other', 'is_sch_holiday']:
    if df_new[col].dtype == 'object':
        df_new[col] = df_new[col].astype(str).str.upper().map({'TRUE': 1, 'FALSE': 0, '1': 1, '0': 0})
    else:
        df_new[col] = df_new[col].astype(int)

# Extract departure month from deaprture date for seasonal features 
df_new['departure_date'] = pd.to_datetime(df_new['departure_date'], errors='coerce')
df_new['departure_month'] = df_new['departure_date'].dt.month

# Check data types
df_new.info()

<class 'pandas.core.frame.DataFrame'>
Index: 480 entries, 1240 to 1719
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               480 non-null    datetime64[ns]
 1   route              480 non-null    object        
 2   departure_date     480 non-null    datetime64[ns]
 3   price              480 non-null    float64       
 4   days_to_departure  480 non-null    int64         
 5   day_of_week        480 non-null    int64         
 6   day_name           480 non-null    object        
 7   is_weekend         480 non-null    int64         
 8   departure_airport  480 non-null    object        
 9   out_inbound        480 non-null    int64         
 10  other_airport      480 non-null    object        
 11  data_source        480 non-null    object        
 12  booking_window     480 non-null    object        
 13  airline            480 non-null    object        
 14  airline_cod

### Feature Selection and Encoding

In [8]:
# Selecting features and one-hot encoding
core_features = [
    'booking_window',
    'day_of_week',
    'is_weekend',
    'is_lcc',
    'is_holiday_sin',
    'is_holiday_other',
    'is_sch_holiday',
    'route',
    'departure_month']

X_tomorrow = pd.get_dummies(df_new[core_features], columns = ['route','booking_window'], drop_first = True)
y_tomorrow = df_new['price']

### Forward Test

In [9]:
# Re-index columns to match trained model
X_tomorrow_align = X_tomorrow.reindex(columns = model_schema, fill_value = 0)

In [12]:
# Execute blind production predictions
predicted_prices = model.predict(X_tomorrow_align)

tomorrow_mae = mean_absolute_error(y_tomorrow, predicted_prices)
tomorrow_rmse = np.sqrt(mean_squared_error(y_tomorrow, predicted_prices))
tomorrow_r2 = r2_score(y_tomorrow, predicted_prices)
tomorrow_mape = mean_absolute_percentage_error(y_tomorrow, predicted_prices)

print(f"Live production Mean Absolute Error (MAE): {tomorrow_mae:.2f} SGD")
print(f"Live production Root Mean Squared Error (RMSE): {tomorrow_rmse:.2f} SGD")
print(f"Live production Variance Coverage (R² Score):   {tomorrow_r2:.4f} ({tomorrow_r2*100:.2f}%)")
print(f"Live production Mean Absolute Percentage Error (MAPE): {tomorrow_mape*100:.2f}%")

Live production Mean Absolute Error (MAE): 221.48 SGD
Live production Root Mean Squared Error (RMSE): 383.42 SGD
Live production Variance Coverage (R² Score):   0.7453 (74.53%)
Live production Mean Absolute Percentage Error (MAPE): 23.98%


#### Analysis

The delta between the training performance $R^{2}$ of 85.3% and live production of 74.5% proves that the model is sensing a market drift. Rather than over-fitting to the past, this variance suggests that the model should be regularly refreshed to keep abreast to the changing airline pricing environment. 